<a href="https://colab.research.google.com/github/taichi0520-hub/my-research-project/blob/main/src/%E4%BD%9C%E6%88%90%E3%83%A2%E3%83%87%E3%83%AB%E3%81%AE%E6%8E%A8%E8%AB%96%E5%AE%9F%E8%A1%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. 準備：ライブラリのインストールとマウント

これは環境を整えるために毎回必要です（T4 GPUで十分高速に動きます）。

In [ ]:
!pip install "cellpose[gui]" --quiet

from google.colab import drive
import os
from cellpose import models, io

drive.mount('/content/drive')

2. 推論の実行（追加学習は不要）

学習済みモデルを指定して、画像を読み込ませるだけです。

Google Driveの左メニューから対象フォルダを右クリックして「パスをコピー」し、このGUIフォームに貼り付ける。

Google Drive からパスをコピーする手順（Colabでのバッチ処理用）
Google Colabの画面左側にある「フォルダアイコン」を使用するのが最も簡単で確実です。

1.左サイドバーを開く

Colabの画面左端にある、「フォルダ」のアイコンをクリックします。

2.Google Driveのマウント確認

drive というフォルダがあるか確認してください。もしなければ、先にマウント用コード（drive.mount('/content/drive')）を実行するか、フォルダアイコンの上部にある「Google ドライブをマウント」のアイコン（フォルダにGoogle Driveのロゴがついたようなマーク）をクリックしてください。

3.対象のフォルダまで移動する

drive > MyDrive > ... と順番にクリックして階層を下り、解析したい画像が入っているフォルダ（例：test_images など）を見つけます。

4.パスをコピーする

対象のフォルダ名（またはファイル名）の右側にある 「縦の3点リーダー（︙）」 をクリックし、メニューから 「パスをコピー」 を選択します。

5.フォームに貼り付け

コピーしたパスを、先ほどのコードの右側にある INPUT_DIR や MODEL_PATH のテキストボックスに貼り付けます。

重要な注意点:
パスの最後（末尾）にスラッシュ（/）がついていなくても、Pythonの os.path.join が適切に処理してくれるため、コピーしたままの文字列を貼り付けて問題ありません。

In [ ]:
# --- 1. 環境構築とマウント ---
!pip install "cellpose[gui]" --quiet
from google.colab import drive
from google.colab import runtime
import os
from cellpose import models, io
import datetime

drive.mount('/content/drive')

# --- 2. パスとパラメータの設定（GUIフォーム） ---
# @title 推論設定
# 以下の変数は、セルの右側に表示される入力ボックスから書き換え可能です。

MODEL_PATH = '/content/drive/MyDrive/cell_analysis/models/my_finetuned_model' #@param {type:"string"}
INPUT_DIR = '/content/drive/MyDrive/cell_analysis/images/mdck_cross_1' #@param {type:"string"}
SAVE_DIR = '/content/drive/MyDrive/cell_analysis/results' #@param {type:"string"}
DIAMETER = 5 #@param {type:"number"}
DISCONNECT_AFTER_RUN = False #@param {type:"boolean"}

# --- 3. 実行処理 ---
# 実行ごとにタイムスタンプ付きのサブフォルダを作成して上書きを防止
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
run_save_dir = os.path.join(SAVE_DIR, f"run_{timestamp}")
os.makedirs(run_save_dir, exist_ok=True)

# エラーハンドリング：パスの存在確認
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ モデルが見つかりません。パスを確認してください: {MODEL_PATH}")
if not os.path.exists(INPUT_DIR):
    raise FileNotFoundError(f"❌ 画像フォルダが見つかりません。パスを確認してください: {INPUT_DIR}")

# モデル読み込み
print("⏳ モデルを読み込んでいます...")
model = models.CellposeModel(gpu=True, pretrained_model=MODEL_PATH)

# 画像ファイルの取得
files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif'))]

if not files:
    print(f"❌ 指定されたフォルダに画像がありません: {INPUT_DIR}")
else:
    print(f"🚀 {len(files)} 枚の画像の推論を開始します...")
    for f in files:
        img = io.imread(os.path.join(INPUT_DIR, f))

        # 推論実行
        masks, flows, styles = model.eval(img, diameter=DIAMETER, channels=[0, 0])

        # 結果の保存 (実行ごとのサブフォルダを指定)
        io.save_masks(img, masks, flows, f, savedir=run_save_dir, tif=True)
        print(f"✅ 完了: {f}")

    print(f"🎉 すべての処理が完了しました。結果は {run_save_dir} に保存されています。")

# --- 4. 実行後のランタイム切断 ---
if DISCONNECT_AFTER_RUN:
    print("🔌 推論が完了したため、ランタイムを切断します。お疲れ様でした！")
    runtime.unassign()


In [ ]:
import os

# SAVE_DIR変数が定義されていることを確認
if 'SAVE_DIR' in locals():
    print(f"SAVE_DIR: {SAVE_DIR}")
    if os.path.exists(SAVE_DIR):
        files_in_save_dir = os.listdir(SAVE_DIR)
        if files_in_save_dir:
            print(f"SAVE_DIR '{SAVE_DIR}' に以下のファイルが見つかりました:")
            for f in files_in_save_dir:
                print(f"- {f}")
        else:
            print(f"SAVE_DIR '{SAVE_DIR}' は空です。")
    else:
        print(f"エラー: SAVE_DIR '{SAVE_DIR}' が存在しません。")
else:
    print("エラー: SAVE_DIR 変数が定義されていません。上記のセルを再度実行してください。")

In [ ]:
import os

# 検索対象のルートディレクトリ（Colabの全体とGoogle Drive）
search_dirs = ['/content']

print("🔍 Cellposeの出力ファイルを検索中...")
found_files = []

for search_dir in search_dirs:
    for root, dirs, files in os.walk(search_dir):
        # 隠しフォルダやシステムフォルダはスキップ
        if '.config' in root or '.Trash' in root:
            continue
        for file in files:
            if 'cp_masks' in file or 'cp_outlines' in file or 'flows' in file:
                full_path = os.path.join(root, file)
                found_files.append(full_path)

if found_files:
    print(f"\n✅ {len(found_files)}個の出力ファイルが見つかりました:")
    for f in found_files:
        print(f)
else:
    print("\n❌ Cellposeの出力ファイルは見つかりませんでした。")


In [ ]:
import os
from google.colab import files

SAVE_DIR = '/content/drive/MyDrive/cell_analysis/results'

print(f"📁 {SAVE_DIR} の中身を確認します...")
if os.path.exists(SAVE_DIR):
    result_files = os.listdir(SAVE_DIR)
    if result_files:
        print("\n以下のファイルが見つかりました:")
        for f in result_files:
            print(f"- {f}")

        # 例として、最初に見つかったPNGファイルをダウンロードします
        png_files = [f for f in result_files if f.endswith('.png')]
        if png_files:
            target_file = os.path.join(SAVE_DIR, png_files[0])
            print(f"\n⬇️ {png_files[0]} をダウンロードします...")
            files.download(target_file)
    else:
        print("フォルダは空です。")
else:
    print("フォルダが存在しません。")